# Text-Timbre Playground

This notebook provides a few methods for playing with systems for generating audio samples.
With this, you can:
- Generate audio from text prompts (text → audio; using Stable Audio Open 1.0)
- Timbre transfer of generated or submitted audio samples (audio → model → audio; uses any model selected from the `RAVE Models` folder, presuming it has a `encode` and `decode` function)
- Interpolate between audio samples (audio1 + audio2 → model → audio; uses linear interpolation in the latent/embedding space of a chosen model)

## How to use

There are a number of ways to run this notebook. You can run it locally, or you might wish to run it on a server solution such as google colab (and to avail yourself of their GPUs).

### Running Locally

### Running on Colab
This repo depends on python ~3.10.14, so, you will need to reset colab to run an older version. You can select this with the `change runtime` option. This notebook has been tested with the 2025.07 runtime.

Then, run the following cells to clone the repo into colab.

# Dependencies

In [2]:
!git clone https://github.com/ashNotKetchup/SAO-Text-Embedding-Playground

Cloning into 'SAO-Text-Embedding-Playground'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 101 (delta 12), reused 14 (delta 6), pack-reused 73 (from 1)
Receiving objects: 100% (101/101), 71.85 MiB | 20.85 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Updating files: 100% (17/17), done.


In [3]:
!pip install -r SAO-Text-Embedding-Playground/requirements.txt

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 69.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 108.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 10.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 11.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 11.0 MB/s eta 0:00:00
  Preparing metadata (setup.py)

In [1]:

!pip install torchcodec
!pip install flash-attn


In [2]:
%cd SAO-Text-Embedding-Playground/
!ls

/content/SAO-Text-Embedding-Playground
 audio_examples   demo.py    'RAVE Models'   requirements.txt
 demo.ipynb	  functions   readme.md      tests


## Hugging Face Login

Some Text to audio models, require huggingface authentication to download.

First, create an account on hugginface ()
then agree to the terms for the appropriate model:
1. Stableaudio()

Please log in to your Hugging Face account if prompted. You will need a Hugging Face token (it starts with `hf_`).

In [3]:
from huggingface_hub import login
# Log in to Hugging Face to access gated models if necessary
login()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Run App

Once you've logged in, run the playground app by pressing play on this cell group

In [ ]:
import gradio as gr
import os



from functions.text_gen import text_to_audio
from functions.latent_interpolation import interpolate_audio


def generate_audio(
    text: str,
    model: str,
    interpolation_mode: str = None,
    interpolation_text: str = None,
    interpolation_audio=None,
    interpolation_amount: float = 0.5,
):
    """Generates audio dependent on input"""
    print(
        f"Received inputs - Text: {text}, Model: {model}, Interpolation Mode: {interpolation_mode}, Interpolation Text: {interpolation_text}, Interpolation Audio: {interpolation_audio}, Interpolation Amount: {interpolation_amount}"
    )
    if interpolation_mode == "Text" and interpolation_text:
        print("Performing text-based interpolation (placeholder)")
        interpolation_audio = text_to_audio(interpolation_text)

    if interpolation_audio:
        return interpolate_audio(
            text_to_audio(text), interpolation_audio, model, interpolation_amount
        )

    return interpolate_audio(text_to_audio(text), model_string=model)


def update_interpolation_inputs(mode: str):
    show_text = mode == "Text"
    show_audio = mode == "Audio"
    return (
        gr.update(visible=show_text),
        gr.update(visible=show_audio),
    )


# Interface
def build_interface():
    # determine model choices from "RAVE Models" folder (next to this script)
    models_dir = os.path.join(os.path.dirname(''), "RAVE Models")
    model_choices = []
    try:
        if os.path.isdir(models_dir):
            # list non-hidden files/directories and create full paths
            model_choices = sorted(
                [
                    os.path.join(models_dir, f)
                    for f in os.listdir(models_dir)
                    if not f.startswith(".")
                ]
            )
    except Exception:
        model_choices = []

    # fallback to defaults if folder missing or empty
    if not model_choices:
        model_choices = ["gpt-4o-mini-tts", "gpt-4o-mini", "gpt-4o"]

    with gr.Blocks() as demo:
        gr.Markdown("# Audio Generation Demo")
        with gr.Row():
            with gr.Column():
                txt = gr.Textbox(
                    label="Input text",
                    placeholder="Enter prompt for audio/music generation",
                )
                model = gr.Dropdown(
                    choices=model_choices, value=model_choices[0], label="Model"
                )
                audio = gr.Audio(label="Audio output")
                btn = gr.Button("Generate")
            with gr.Column():
                interp_mode = gr.Dropdown(
                    choices=["Text", "Audio"],
                    value="Text",
                    label="Interpolation source",
                )
                interp_txt = gr.Textbox(
                    label="Interpolation text (optional)",
                    placeholder="Optional text to interpolate from",
                    visible=True,
                )
                interp_audio = gr.Audio(
                    label="Interpolation audio (optional)",
                    sources=["upload"],
                    type="filepath",
                    visible=False,
                )
                interp_amount = gr.Slider(
                    minimum=0.0,
                    maximum=1.0,
                    value=0.5,
                    step=0.05,
                    label="Interpolation amount",
                )

        interp_mode.change(
            fn=update_interpolation_inputs,
            inputs=interp_mode,
            outputs=[interp_txt, interp_audio],
        )
        btn.click(
            fn=generate_audio,
            inputs=[txt, model, interp_mode, interp_txt, interp_audio, interp_amount],
            outputs=audio,
        )

    return demo


if __name__ == "__main__":
    demo = build_interface()
    demo.launch(debug=True)

In [ ]:
import torch
import torchaudio
!pip install torchcodec
output = torch.randn(2,44100)
torchaudio.save("test.wav", output, 44100)


In [ ]:
!ls

tests

In [8]:
import tempfile
import os
import soundfile as sf
import torch
import torchaudio

# From Stable audio open page on huggingface (https://huggingface.co/stabilityai/stable-audio-open-1.0):

import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
from stable_audio_tools.inference.generation import generate_diffusion_cond

PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

def text_to_audio(text: str, length_secs: int = 15) -> str:
    """Audio generator: create a WAV file from text -> audio model and return its path."""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Clear CUDA cache before loading the model to free up GPU memory
    if device == "cuda":
      torch.cuda.empty_cache()
    # Download model
    model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
    sample_rate = model_config["sample_rate"]
    sample_size = model_config["sample_size"]

    model = model.to(device)

    # Set up text and timing conditioning
    conditioning = [
        {
            "prompt": text,
            "negative_prompt": "low quality and distorted",
            "seconds_start": 0,
            "seconds_total": length_secs,
        }
    ]

    # Generate stereo audio
    output = generate_diffusion_cond(
        model,
        steps=150,
        cfg_scale=7,
        conditioning=conditioning,
        sample_size=sample_size,
        sigma_min=0.3,
        sigma_max=500,
        sampler_type="dpmpp-3m-sde",
        device=device,
    )

    # Rearrange audio batch to a single sequence
    output = rearrange(output, "b d n -> d (b n)")

    # Peak normalize, clip, convert to float 32, and save to file
    output = (
        output.to(torch.float32)
        .div(torch.max(torch.abs(output)))
        .clamp(-1, 1)
        .mul(32767)
        # .to(torch.int16)
        .cpu()[:, : (length_secs * sample_rate)]  # Trim to length_secs
    )

    # torchaudio.save(path, output, sample_rate)

    return output

audio_path = text_to_audio("4/4 drum groove")


/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

1790753523


/usr/local/lib/python3.11/dist-packages/stable_audio_tools/models/conditioners.py:362: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16) and torch.set_grad_enabled(self.enable_grad):


  0%|          | 0/150 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torchsde/_brownian/brownian_interval.py:608: UserWarning: Should have tb<=t1 but got tb=500.00006103515625 and t1=500.000061.
  warnings.warn(f"Should have {tb_name}<=t1 but got {tb_name}={tb} and t1={self._end}.")


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 749.81 MiB is free. Including non-PyTorch memory, this process has 13.83 GiB memory in use. Of the allocated memory 13.68 GiB is allocated by PyTorch, and 18.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
import IPython.display as Audio

Audio(audio_path)

In [ ]:
!pip install stable_audio_tools

In [ ]:
from huggingface_hub import login
# Log in to Hugging Face to access gated models if necessary
login()

In [4]:
import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
from stable_audio_tools.inference.generation import generate_diffusion_cond

device = "cuda" if torch.cuda.is_available() else "cpu"

# Download model
model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
sample_rate = model_config["sample_rate"]
sample_size = model_config["sample_size"]

model = model.to(device)

# Set up text and timing conditioning
conditioning = [{
    "prompt": "jazz flute",
    "negative_prompt": "low quality and distorted",
    "seconds_start": 0,
    "seconds_total": 30
}]

# Generate stereo audio
output = generate_diffusion_cond(
    model,
    steps=100,
    cfg_scale=7,
    conditioning=conditioning,
    sample_size=sample_size,
    sigma_min=0.3,
    sigma_max=500,
    sampler_type="dpmpp-3m-sde",
    device=device
)

# Rearrange audio batch to a single sequence
output = rearrange(output, "b d n -> d (b n)")

# Peak normalize, clip, convert to int16, and save to file
output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).mul(32767).to(torch.int16).cpu()
torchaudio.save("output.wav", output, sample_rate)


model_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/flash_attn_2_cuda.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib
flash_attn not installed, disabling Flash Attention


/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.85G [00:00<?, ?B/s]

1685479614


/usr/local/lib/python3.11/dist-packages/stable_audio_tools/models/conditioners.py:362: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16) and torch.set_grad_enabled(self.enable_grad):


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torchsde/_brownian/brownian_interval.py:608: UserWarning: Should have tb<=t1 but got tb=500.00006103515625 and t1=500.000061.
  warnings.warn(f"Should have {tb_name}<=t1 but got {tb_name}={tb} and t1={self._end}.")


In [5]:
import tempfile
import os
import soundfile as sf
import torch
import torchaudio
import IPython.display as Audio

# From Stable audio open page on huggingface (https://huggingface.co/stabilityai/stable-audio-open-1.0):

import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
from stable_audio_tools.inference.generation import generate_diffusion_cond


def text_to_audio(text: str, length_secs: int = 15) -> str:
    """Audio generator: create a WAV file from text -> audio model and return its path."""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Clear CUDA cache before loading the model to free up GPU memory
    if device == "cuda":
        torch.cuda.empty_cache()

    # Download model
    model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
    sample_rate = model_config["sample_rate"]
    sample_size = model_config["sample_size"]

    model = model.to(device)

    # Set up text and timing conditioning
    conditioning = [
        {
            "prompt": text,
            "seconds_start": 0,
            "seconds_total": length_secs,
        }
    ]

    # Generate stereo audio
    output = generate_diffusion_cond(
        model,
        steps=100,
        cfg_scale=7,
        conditioning=conditioning,
        sample_size=sample_size,
        sigma_min=0.3,
        sigma_max=500,
        sampler_type="dpmpp-3m-sde",
        device=device,
    )

    # Rearrange audio batch to a single sequence
    output = rearrange(output, "b d n -> d (b n)")

    # Peak normalize, clip, convert to float 32, and save to file
    output = (
        output.to(torch.float32)
        .div(torch.max(torch.abs(output)))
        .clamp(-1, 1)
        .mul(32767)
        .to(torch.int16) # Changed to int16 for torchaudio.save compatibility with common audio formats
        .cpu()[:, : (length_secs * sample_rate)]  # Trim to length_secs
    )

    # Create a temporary file to save the audio
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
        path = tmp_file.name

    torchaudio.save(path, output, sample_rate)
    # Audio(path)
    return 'output.wav'

audio_path = text_to_audio("4/4 drum groove", length_secs=5)
print(f"Audio saved to: {audio_path}")

/usr/local/lib/python3.11/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/99 [00:00<?, ?it/s]

3482582024


/usr/local/lib/python3.11/dist-packages/stable_audio_tools/models/conditioners.py:362: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16) and torch.set_grad_enabled(self.enable_grad):


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torchsde/_brownian/brownian_interval.py:608: UserWarning: Should have tb<=t1 but got tb=500.00006103515625 and t1=500.000061.
  warnings.warn(f"Should have {tb_name}<=t1 but got {tb_name}={tb} and t1={self._end}.")


OutOfMemoryError: CUDA out of memory. Tried to allocate 1024.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 747.81 MiB is free. Including non-PyTorch memory, this process has 13.83 GiB memory in use. Of the allocated memory 13.68 GiB is allocated by PyTorch, and 20.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [5]:
pip install flash-attn --no-build-isolation